In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import os
import torch
import json
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel

def clean_adapter_config(adapter_path):
    """Fungsi pembantu untuk membersihkan config JSON seperti kode Anda sebelumnya"""
    config_file = os.path.join(adapter_path, "adapter_config.json")
    if os.path.exists(config_file):
        with open(config_file, 'r') as f:
            config_data = json.load(f)
        if "alora_invocation_tokens" in config_data:
            del config_data["alora_invocation_tokens"]
            with open(config_file, 'w') as f:
                json.dump(config_data, f, indent=2)

def load_bilingual_models(adapter_path_id, adapter_path_en, base_model_name="Salesforce/blip2-flan-t5-xl"):
    print("1. Memuat Processor...")
    processor = Blip2Processor.from_pretrained(base_model_name, use_fast=False)

    print(f"2. Memuat Base Model: {base_model_name} (Hanya dilakukan 1 kali)...")
    base_model = Blip2ForConditionalGeneration.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    print("3. Memperbaiki Config Adapter...")
    clean_adapter_config(adapter_path_id)
    clean_adapter_config(adapter_path_en)

    print(f"4. Memasang Adapter Bahasa Indonesia dari: {adapter_path_id}...")
    model = PeftModel.from_pretrained(base_model, adapter_path_id, adapter_name="indo")
    
    print(f"5. Memasang Adapter Bahasa Inggris dari: {adapter_path_en}...")
    model.load_adapter(adapter_path_en, adapter_name="eng")
    
    print("Semua model berhasil dimuat dan digabungkan!\n")
    return model, processor

def generate_caption(model, processor, image_path, prompt, adapter_name):
    """Fungsi helper untuk generate satu bahasa"""
    device = model.device
    
    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        return f"Error: Tidak dapat membuka gambar di {image_path}. Detail: {e}"

    model.set_adapter(adapter_name)

    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_length": 300,        
        "min_length": 30,         
        "num_beams": 5,           
        "repetition_penalty": 1.2,
        "no_repeat_ngram_size": 3, 
        "do_sample": False,        
    }
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, **gen_kwargs)
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return caption.replace(prompt, "").strip()

def run_bilingual_inference(model, processor, image_path):
    """Menjalankan kedua bahasa secara bergantian dengan sangat efisien"""
    model.eval()
    
    prompt_id = "Deskripsikan batik ini secara komprehensif, meliputi nama motif, elemen visual, warna, teknik pembuatan, dan makna filosofisnya: "
    prompt_en = "Describe this batik comprehensively, including the motif name, visual elements, colors, crafting technique, and its philosophical meaning: "

    print("Memproses Bahasa Indonesia...")
    caption_id = generate_caption(model, processor, image_path, prompt_id, adapter_name="indo")

    print("Memproses Bahasa Inggris...")
    caption_en = generate_caption(model, processor, image_path, prompt_en, adapter_name="eng")

    return {
        "id": caption_id,
        "en": caption_en
    }

/mnt/extended-home/dzakaaufa/dzakanenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if __name__ == "__main__":
    ADAPTER_ID = "/mnt/extended-home/dzakaaufa/models/flan-t5-xl-indo"
    ADAPTER_EN = "/mnt/extended-home/dzakaaufa/models/flan-t5-xl-eng"
    TEST_IMAGE = "/mnt/extended-home/dzakaaufa/dataset/image/Data 32 Batik Soendari (Bunga Apel).jpg" 

    model, processor = load_bilingual_models(ADAPTER_ID, ADAPTER_EN)

    hasil = run_bilingual_inference(model, processor, TEST_IMAGE)
    
    print("\n" + "="*50)
    print("HASIL DESKRIPSI BAHASA INDONESIA:")
    print(hasil["id"])
    print("\n" + "="*50)
    print("HASIL DESKRIPSI BAHASA INGGRIS:")
    print(hasil["en"])
    print("="*50)

1. Memuat Processor...


/mnt/extended-home/dzakaaufa/dzakanenv/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


2. Memuat Base Model: Salesforce/blip2-flan-t5-xl (Hanya dilakukan 1 kali)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:27<00:00, 13.62s/it]


3. Memperbaiki Config Adapter...
4. Memasang Adapter Bahasa Indonesia dari: /mnt/extended-home/dzakaaufa/models/flan-t5-xl-indo...
5. Memasang Adapter Bahasa Inggris dari: /mnt/extended-home/dzakaaufa/models/flan-t5-xl-eng...
Semua model berhasil dimuat dan digabungkan!

Memproses Bahasa Indonesia...
Memproses Bahasa Inggris...

HASIL DESKRIPSI BAHASA INDONESIA:
batik junjung derajat mengangkat tema culture, yaitu topeng/mask,. dibuat dengan teknik tulis, batik ini menonjolkan perpaduan warna biru/blue, putih/white di atas kain primisima. motifnya yang berbentuk non geometris menggunakan pewarna alam. terinspirasi dari sumber mata air polaman, kecamatan dan kekayaan pada kebudayaan tionghoa

HASIL DESKRIPSI BAHASA INGGRIS:
kupu-kupu sulur merupakan mahakarya dari kelas batik malang yang mengusung pola non geometris. secara visual, desain kain ini didominasi oleh perpaduan biru tua, kuning, dan coklat dan merah dan hitam dan putih, melambangkan kesuburan dan keindahan. karya ini dibuat 